In [1]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler,LabelEncoder
from sklearn.metrics import classification_report,f1_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

In [2]:
df = pd.read_csv('intrusion_dataset_features.csv')
df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,wrong_fragment,urgent,count,srv_count,label
0,56.081581,tcp,ftp,RSTR,3891.938313,905.010282,0,0,6,4,Normal
1,3.851167,udp,dns,SF,1629.268860,1509.760944,0,2,31,41,BruteForce
2,3.680515,tcp,dns,S0,10131.875360,3148.821581,1,0,194,92,DoS
3,2.886041,icmp,ftp,S0,21202.941391,1468.930855,2,0,120,143,DoS
4,28.101916,udp,smtp,RSTR,4674.627374,4394.753286,0,0,18,3,Normal


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   duration        100000 non-null  float64
 1   protocol_type   100000 non-null  object 
 2   service         100000 non-null  object 
 3   flag            100000 non-null  object 
 4   src_bytes       100000 non-null  float64
 5   dst_bytes       100000 non-null  float64
 6   wrong_fragment  100000 non-null  int64  
 7   urgent          100000 non-null  int64  
 8   count           100000 non-null  int64  
 9   srv_count       100000 non-null  int64  
 10  label           100000 non-null  object 
dtypes: float64(3), int64(4), object(4)
memory usage: 8.4+ MB


In [4]:
df['label'].value_counts()

label
Normal        49934
DoS           25121
Probe         15119
BruteForce     9826
Name: count, dtype: int64

In [5]:
df['label'].replace({'Normal':0,'DoS':1,'Probe':1,'BruteForce':1},inplace=True)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   duration        100000 non-null  float64
 1   protocol_type   100000 non-null  object 
 2   service         100000 non-null  object 
 3   flag            100000 non-null  object 
 4   src_bytes       100000 non-null  float64
 5   dst_bytes       100000 non-null  float64
 6   wrong_fragment  100000 non-null  int64  
 7   urgent          100000 non-null  int64  
 8   count           100000 non-null  int64  
 9   srv_count       100000 non-null  int64  
 10  label           100000 non-null  int64  
dtypes: float64(3), int64(5), object(3)
memory usage: 8.4+ MB


In [7]:
x = df.drop(columns=['label'],axis=1)
y = df['label']

In [8]:
cat_col = df.select_dtypes(include='object').columns
cat_col

Index(['protocol_type', 'service', 'flag'], dtype='object')

In [9]:
xtrain, xtest, ytrain, ytest = train_test_split(x,y,train_size=0.8,random_state=42)

In [10]:
encoder = OneHotEncoder(sparse_output=False,handle_unknown='ignore')

In [11]:
cat_val = encoder.fit_transform(xtrain[cat_col])
cat_val

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 1., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 1.],
       [0., 0., 1., ..., 0., 0., 1.],
       [0., 1., 0., ..., 0., 0., 1.]])

In [12]:
xtrain.drop(columns=cat_col,axis=1,inplace=True)

In [13]:
xtrain[encoder.get_feature_names_out()] = cat_val
xtrain.head()

,duration,src_bytes,dst_bytes,wrong_fragment,urgent,count,srv_count,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,service_dns,service_ftp,service_http,service_smtp,service_ssh,flag_REJ,flag_RSTR,flag_S0,flag_SF
75220,27.559036,3305.917994,4017.072694,0,0,4,14,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
48955,39.191208,5972.728995,4590.354204,1,0,6,13,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
44966,43.446759,3567.325059,4713.844923,0,0,11,11,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
13568,2.510376,2374.551898,1747.472957,0,1,31,18,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
92727,46.328624,2746.281596,2833.723517,1,0,11,3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [14]:
cat_val_test = encoder.transform(xtest[cat_col])
cat_val_test

array([[0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.]])

In [15]:
xtest.drop(columns=cat_col,axis=1,inplace=True)

In [16]:
xtest[encoder.get_feature_names_out()] = cat_val_test
xtest.head()

,duration,src_bytes,dst_bytes,wrong_fragment,urgent,count,srv_count,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,service_dns,service_ftp,service_http,service_smtp,service_ssh,flag_REJ,flag_RSTR,flag_S0,flag_SF
75721,27.578990,3786.567675,4005.195585,0,0,4,6,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
80184,30.364423,5126.219348,3468.365770,0,0,8,5,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
19864,10.777566,3801.505571,1675.306186,0,0,60,27,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
76699,13.937267,3611.360063,2145.364079,4,0,65,27,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
92991,17.521132,4793.741536,2006.011552,0,0,14,3,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [17]:
model = RandomForestClassifier(random_state=42)

In [18]:
model.fit(xtrain,ytrain)

RandomForestClassifier(random_state=42)

In [19]:
y_pred = model.predict(xtrain)
y_pred

array([0, 0, 0, ..., 0, 1, 0], dtype=int64)

In [20]:
print(classification_report(ytrain,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     40039
           1       1.00      1.00      1.00     39961

    accuracy                           1.00     80000
   macro avg       1.00      1.00      1.00     80000
weighted avg       1.00      1.00      1.00     80000



In [21]:
y_pred_test = model.predict(xtest)
y_pred_test

array([0, 0, 1, ..., 1, 0, 1], dtype=int64)

In [22]:
print(classification_report(ytest,y_pred_test))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9895
           1       1.00      1.00      1.00     10105

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000



In [23]:
joblib.dump(encoder,'encoder.pkl')
joblib.dump(model,'model.pkl')

['model.pkl']

In [24]:
import sklearn
print(sklearn.__version__)

1.4.2
